# GeoSR-4 — Synthetic-degradation pretraining (D048)

Tests whether pretraining on a synthetic NAIP-degradation corpus (opensr-degradation, real HR-only NAIP tiles
turned into synthetic Sentinel-2-like LR) helps before fine-tuning on the real SEN2NAIP pairs, vs training on
real data alone.

**Fair comparison target: D040's 7a (ICNR-only, --amp, 20 real epochs, PSNR 16.54 dB)** -- same ICNR default,
same `--amp`, same 20-epoch finetune count, batch-size 16, embed_dim 60. Only the synthetic-pretrain phase
prepended before it is new, so any difference is attributable to that.

**Honest scale caveat**: the synthetic corpus is only 20 locations right now (proof-of-concept scale, not yet
the "abundant synthetic data" scale the technique is usually used at) -- generated fresh in this notebook from
real NAIP HR tiles across 20 diverse US locations via Microsoft Planetary Computer (free, no auth), degraded
with `opensr-degradation`'s NAIP→Sentinel-2-like model. Scaling this up later just means fetching more
locations (`ml/datasets/generate_synthetic_pairs.py --n <bigger>`).

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision opensr-degradation einops datasets pystac-client planetary-computer

## 2. Download the real SEN2NAIP dataset (needed for fine-tuning + validation)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Generate the synthetic pretraining corpus
Fetches real HR-only NAIP tiles from 20 diverse US locations (free, no auth, via Microsoft Planetary Computer)
and degrades each into a synthetic (LR, HR) pair. Takes a few minutes (network + CPU-bound degradation model).
To use more locations, add more `(lon, lat)` tuples to `LOCATIONS` in `ml/datasets/generate_synthetic_pairs.py`
and pass a bigger `--n`.

In [ ]:
!python ml/datasets/generate_synthetic_pairs.py --n 20

## 4. Pretrain on synthetic corpus, then fine-tune on real data
Phase 1 (pretrain) evaluates on the *real* val split throughout, so you can watch it converge toward real-data
performance even while only training on synthetic pairs. Phase 2 (fine-tune) continues the same model on the
real train split -- same protocol as D040's 7a (20 epochs, batch 16, `--amp`) for a fair comparison.

In [ ]:
!python ml/training/train_swinir_synthetic_pretrain.py \
  --pretrain-epochs 40 \
  --finetune-epochs 20 \
  --batch-size 4 \
  --finetune-batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/swinir_synthetic_pretrain \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_synthetic_pretrain/swinir_finetune_epoch19.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Download the final checkpoint

In [ ]:
from google.colab import files
files.download('experiments/swinir_synthetic_pretrain/swinir_finetune_epoch19.pt')

## 6. Optional: bigger model + longer training (run instead of section 4, not in addition)
Combines the other queued backlog item ("bigger/longer retrain") with the same synthetic-pretrain pipeline --
`embed_dim` 60→120 and RSTB depths 4 blocks→6 blocks each (roughly 2-3x the capacity of every run so far in
this project), pretrain-epochs 40→60, finetune-epochs 20→40. This is a genuinely bigger commitment: expect
several hours on a T4, and there's no prior run at this capacity to sanity-check against -- watch the first
few epochs' loss for NaN/divergence before walking away.

In [ ]:
!python ml/training/train_swinir_synthetic_pretrain.py \
  --pretrain-epochs 60 \
  --finetune-epochs 40 \
  --batch-size 4 \
  --finetune-batch-size 16 \
  --embed-dim 120 \
  --depths 6,6,6,6 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/swinir_bigger_synthetic_pretrain \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_bigger_synthetic_pretrain/swinir_finetune_epoch39.pt \
  --model-type swinir --embed-dim 120 --depths 6,6,6,6 --num-heads 6 --window-size 11